# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KijoSal-dev/flyrank-ml-internship-wk1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [14]:
# connecting HuggingFace and DuckDB
import getpass
import duckdb

# Enter your Hugging Face READ token when prompted.
# The token will not be displayed or saved in this notebook.
HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token (hf_...): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError("That does not look like a Hugging Face token.")

# Connect DuckDB
con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Hugging Face warehouse location
REL = "hf://datasets/FlyRank/internship-warehouse"

# Warehouse tables
TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Hugging Face token accepted.")
print("DuckDB connected.")
print("FlyRank warehouse paths configured.")


Hugging Face token accepted.
DuckDB connected.
FlyRank warehouse paths configured.


In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: verify the unit of analysis and March 2026 time window

grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain combinations found:", len(grain_check))

window_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

window_check


Duplicate grain combinations found: 0


,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


## 1. Unit of analysis + time window

One row represents one content item for one client on one report date.

I use March 2026 as the development month because it is a mid-panel month rather than the final June 2026 month. The observed March 2026 window contains 9,841,378 rows, with report dates from 2026-03-01 to 2026-03-31.

The grain check found 0 duplicate combinations of report date, client, and content in March 2026. This supports the stated grain for this development month.

June 2026 is treated as a sealed final month and is not used to develop the label or features.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [16]:
# Section 2: inspect the available fields in the daily performance table

columns = con.sql(f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    LIMIT 0
""").df()

print("Available columns:")
for col in columns.columns:
    print(col)



Available columns:
report_date
client_hash_id
content_hash_id
client_has_gsc
client_has_ga4
gsc_data_available
ga4_data_available
gsc_impressions
gsc_clicks
gsc_sum_position
gsc_avg_position
ga4_pageviews
ga4_sessions
ga4_users
ga4_engaged_sessions
ga4_total_engagement_sec
sessions_organic
sessions_direct
sessions_referral
sessions_social
sessions_paid
sessions_ai
ai_chatgpt
ai_perplexity
ai_gemini
ai_copilot
ai_claude
ai_meta
ai_other
scroll_events
month


In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

selected_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_engaged_sessions",
    "sessions_organic"
]

missing_features = [
    col for col in selected_features
    if col not in columns.columns
]

print("Selected features:", selected_features)
print("Missing features:", missing_features)


Selected features: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_engaged_sessions', 'sessions_organic']
Missing features: []


## 2. Fields: feature / label / context / excluded

The field check confirmed that all five selected features are present in the daily performance table. The check returned an empty missing-features list.

The five features are:
- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_engaged_sessions`
- `sessions_organic`

These will be used as features only when their values are available before the future outcome being predicted.

The future-period search performance will be treated as the label/outcome and will not be used as a feature.

I treat `report_date`, `month`, `client_hash_id`, and `content_hash_id` as context fields because they identify the observation, time period, client, or content rather than serving as predictive features.

The data-availability fields (`client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available`) are context fields used to understand whether the underlying measurements are available.

I exclude future-period metrics because they would not be knowable at the decision moment. I also exclude `gsc_sum_position` because `gsc_avg_position` is the position measure selected for this analysis. The first feature frame is limited to five features as required by the assignment.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check


,march_rows,ga4_available_rows
0,9841378,413966


## 3. Verify it with queries (grain, counts, missing values, windows)

The March 2026 verification measured 9,841,378 daily performance rows. The grain check found 0 duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id`, supporting the stated grain of one content item for one client on one report date.

The March window runs from 2026-03-01 through 2026-03-31.

For availability, 413,966 of the 9,841,378 March rows have `ga4_data_available IS TRUE`, which is about 4.21% of the March rows. This shows that GA4 coverage is limited in this slice. I therefore will not interpret GA4 zeros as zero engagement when `ga4_data_available` is FALSE.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

client_history = con.sql(f"""
    SELECT
        COUNT(*) AS total_clients,
        COUNT(gsc_data_start) AS clients_with_gsc_start,
        MIN(gsc_data_start) AS earliest_gsc_start,
        MAX(gsc_data_start) AS latest_gsc_start,
        COUNT(ga4_data_start) AS clients_with_ga4_start,
        MIN(ga4_data_start) AS earliest_ga4_start,
        MAX(ga4_data_start) AS latest_ga4_start
    FROM {TABLES['dim_clients']}
""").df()

client_history


,total_clients,clients_with_gsc_start,earliest_gsc_start,latest_gsc_start,clients_with_ga4_start,earliest_ga4_start,latest_ga4_start
0,104,67,2025-01-27,2026-06-02,51,2025-10-29,2026-06-01


## 4. Data limits

The client history check shows that the warehouse is an unbalanced panel. There are 104 clients in total, but only 67 have a recorded GSC data start date and 51 have a recorded GA4 data start date.

GSC history starts as early as 2025-01-27 and as late as 2026-06-02. GA4 history starts as early as 2025-10-29 and as late as 2026-06-01.

This means the data cannot support the assumption that every client has the same amount of historical search or analytics data. A missing value or zero should not automatically be interpreted as zero activity.

For this assignment, I use March 2026 as the development window and treat June 2026 as a sealed final month. I will also use the data-availability flags when interpreting GA4 and GSC measurements.

A key limitation of this slice is that clients with shorter histories provide less historical information for feature construction and comparison.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.